## Prep

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

import warnings
warnings.filterwarnings("ignore")

In [ ]:
fb_data = pd.read_excel('Daily Show Advertising data.xlsx', sheet_name='All data')

fb_data[['Ad_type', 'Ad_content', 'na', 'na1']] = fb_data['Ad name'].str.split('|', expand=True)
fb_data = fb_data.drop(['na', 'na1'], axis=1)

# fb_data['Starts'] = pd.to_datetime(fb_data['Starts'])
# fb_data['Ends'] = pd.to_datetime(fb_data['Ends'])
fb_data['Day'] = pd.to_datetime(fb_data['Day'])

ad_data = fb_data.drop(['Ad name', 'Attribution setting', 'Starts', 'Ends', 'Reporting starts', 'Reporting ends'], axis=1)
ad_data = ad_data[['Ad_type', 'Ad_content', 'Day', 'Amount spent (USD)']]
ad_data.columns = ['type', 'performance', 'date', 'ad spent']

ad_data = ad_data[~ad_data['performance'].isna()]
ad_data = ad_data.groupby('date')['ad spent'].sum().reset_index()

ad_data['ad spent^2'] = ad_data['ad spent'] ** 2

# ad_data['original date'] = ad_data['date']
ad_data['date'] = ad_data['date'] - timedelta(days=30)

ad_data

In [ ]:
ticket_sales = pd.read_excel('SSO Report Jan 2020 thru Oct 2023.xlsx')
memb_hist = pd.read_excel("Member History (1).xlsx", sheet_name='MemHistory')
sf_zipcodes = pd.read_excel('San_Francisco_ZIP_Codes.xlsx', sheet_name='zip_codes')

In [ ]:
# streaming members
subs = memb_hist[memb_hist['Membership Level'].isin(
    ['Annual Digital', 'Corporate Annual Digital', 'Corporate One Month Digital', 'Monthly Digital', 'One Month Digital', 'Three Month Digital Membership'])]
subs = subs.sort_values(
    ['Constituent ID', 'Initiation Date'], ascending=[True, False])

# memb_lvl
# subs = subs[['Constituent ID', 'Membership Level', 'Initiation Date', 'Expiration Date']]
# subs['memb_lvl'] = subs['Membership Level'].apply(lambda x: True if 'Month' in x else False)
mem_cols = pd.get_dummies(
    subs['Membership Level'], drop_first=True, prefix='membership').columns
subs = pd.get_dummies(
    subs, columns=['Membership Level'], drop_first=True, prefix='membership')

# join
subs = pd.merge(ticket_sales, subs, how='left',
                left_on='customer_no', right_on='Constituent ID')
subs[mem_cols] = subs[mem_cols].fillna(False)

# # customers living in SF


def convert_to_int(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return 99999


subs['zip'] = subs['Primary Address Postcode'].apply(lambda x: str(x)[:5])
subs['zip'] = subs['zip'].apply(lambda x: '99999' if len(x) < 5 else x)
subs['zip'] = subs['zip'].fillna('99999')
subs['zip'] = subs['zip'].apply(convert_to_int)
subs['% customers living in SF / total'] = subs['zip'].isin(sf_zipcodes['Zip Code'])
# subs['# purchases'] = subs['zip'].isin(sf_zipcodes['Zip Code'])

# drop cols
drop_columns = ['paid_flag', 'printed_flag', 'seated_flag', 'created_by', 'last_update_by', 'last_updated_dt', 'location', 'account_no',
                'card_expiry_dt', 'act_name', 'description', 'perf_dt', 'mos_desc', 'ord_notes', 'memb_level', 'tot_bal_amt', 'delivery_desc', 'channel']
subs = subs.drop(drop_columns, axis=1).reset_index(drop=True)

# sub
subs['create_dt'] = pd.to_datetime(
    subs['create_dt'], format='%m/%d/%Y %H:%M:%S:%f').dt.date
# subs['sub'] = (subs['create_dt'] >= subs['Initiation Date']) & (
#     subs['create_dt'] <= subs['Expiration Date'])
subs['date'] = subs['create_dt']

# mos
subs = subs[subs['mos'] != 11]
subs['% online purchases / total'] = subs['mos'].apply(
    lambda x: True if x in (3, 10, 46) else False)

# sales
subs['live shows ticket sales'] = subs['tot_due_amt']
subs = subs[subs['live shows ticket sales'] > 0]

# num_seats
subs['% multiple seats purchase / total'] = subs['num_seats'].apply(
    lambda x: True if x > 1 else False)

# subs = subs[['create_dt', 'customer_no', 'order_no', 'perf_code', 'live shows ticket sales', 'memb_lvl', 'sub', '% online purchases / total', '% multiple seats purchase / total', '# customers living in SF']]
subs

In [ ]:
subs = subs[['date', 'live shows ticket sales', '% online purchases / total',
             '% multiple seats purchase / total', '% customers living in SF / total'] + mem_cols.to_list()]

sales = subs.groupby('date').agg({
    'live shows ticket sales': 'sum',
    'membership_Corporate Annual Digital': 'sum',
    'membership_Corporate One Month Digital': 'sum',
    'membership_Monthly Digital': 'sum',
    'membership_One Month Digital': 'sum',
    'membership_Three Month Digital Membership': 'sum',
    '% online purchases / total': lambda x: x.sum() / len(x) * 100,
    '% multiple seats purchase / total': lambda x: x.sum() / len(x) * 100,
    '% customers living in SF / total': lambda x: x.sum() / len(x) * 100,
    # '# purchases': 'count'
}).reset_index()

sales['date'] = pd.to_datetime(sales['date'])

sales

In [ ]:
df = pd.merge(ad_data, sales, on='date')

# quarter
quarters_dummies = pd.get_dummies(df['date'].dt.quarter, prefix='quarter', drop_first=True, dtype=int)
df = pd.concat([df, quarters_dummies], axis=1)

# weekday or weekend
df['weekday_or_weekend'] = np.where(df['date'].dt.day_of_week < 5, 0, 1)

# monthly membership
df['membership_Monthly Digital'] = df['membership_Monthly Digital'] + df['membership_One Month Digital']
df = df.drop(['membership_One Month Digital'], axis=1)

# renaming columns
cols = [x.replace('_', ' ') for x in df.columns]
df.columns = cols

df.head()

In [ ]:
# df.to_excel('df.xlsx', index=False)

## Models

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.ensemble import VotingRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
df.describe()

### Ad spent

In [ ]:
X = df.drop(['date', 'live shows ticket sales', 'ad spent^2',
            'membership Corporate Annual Digital', 'membership Corporate One Month Digital', 
            'membership Three Month Digital Membership'], axis=1)

y = df['live shows ticket sales']

df.describe()

In [ ]:
X.corr()

In [ ]:
import statsmodels.api as sm

# split
# X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=666)

# scale the data
binary_cols = ['quarter 2', 'quarter 3', 'quarter 4', 'weekday or weekend']
cols_to_scale = list(set(X.columns) - set(binary_cols))
X_scaled = X.copy()
# X_test_scaled = X_test.copy()
scaler = StandardScaler()
X_scaled[cols_to_scale] = scaler.fit_transform(X_scaled[cols_to_scale])
# X_test_scaled[cols_to_scale] = scaler.transform(X_test_scaled[cols_to_scale])

# OLS
X_scaled_with_intercept = sm.add_constant(X_scaled)
ols_model = sm.OLS(y, X_scaled_with_intercept).fit()

ols_model.summary()

In [ ]:
vif = pd.DataFrame()
vif["Features"] = X_scaled_with_intercept.columns
vif["VIF"] = [variance_inflation_factor(X_scaled_with_intercept, i) for i in range(X_scaled_with_intercept.shape[1])]

print(vif)

In [ ]:
X = df.drop(['date', 'live shows ticket sales', 'ad spent',
            'membership Corporate Annual Digital', 'membership Corporate One Month Digital', 
            'membership Three Month Digital Membership'], axis=1)

y = df['live shows ticket sales']

# scale the data
binary_cols = ['quarter 2', 'quarter 3', 'quarter 4', 'weekday or weekend']
cols_to_scale = list(set(X.columns) - set(binary_cols))
X_scaled = X.copy()
# X_test_scaled = X_test.copy()
scaler = StandardScaler()
X_scaled[cols_to_scale] = scaler.fit_transform(X_scaled[cols_to_scale])
# X_test_scaled[cols_to_scale] = scaler.transform(X_test_scaled[cols_to_scale])

# OLS
X_scaled_with_intercept = sm.add_constant(X_scaled)
ols_model = sm.OLS(y, X_scaled_with_intercept).fit()

ols_model.summary()